# Simple Campus Menu Agent

This notebook creates a simple agent that can answer questions about today's campus menu.
It uses the `get_menu` tool from `getmenus.py` to look up data from the database.

In [21]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from IPython.display import display, Markdown
from loguru import logger

# Import the get_menu tool and helper functions from getmenus.py
from getmenus import get_menu, get_today, setup_logging

setup_logging()
load_dotenv()
print("Environment loaded.")

2026-06-04 04:25:06.174 | INFO     | getmenus:setup_logging:47 - Logging to /workspaces/hai5016-project/logs/menu_agent_2026-06-04.log


Environment loaded.


In [22]:
logger.info(f"Today's date (Seoul time): {get_today()}")

2026-06-04 04:25:06.182 | INFO     | __main__:<module>:1 - Today's date (Seoul time): 2026-06-04


## Create the Language Model

Connect to Azure OpenAI using credentials stored in `.env`.

In [23]:
# Create the language model using Azure OpenAI credentials from the .env file
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

logger.info("LLM ready.")


2026-06-04 04:25:06.192 | INFO     | __main__:<module>:11 - LLM ready.


## Build the Agent

Create the agent with the `get_menu` tool and a helpful system prompt.

In [24]:
today = get_today()

# The system prompt tells the agent how to behave and what today's date is
SYSTEM_PROMPT = (
    f"You are a helpful campus menu assistant. Today's date is {today} (Asia/Seoul). "
    "Use the get_menu tool to fetch menu data for the requested date. "
    "Translate all Korean names (universities, restaurants, dishes) to English. "
    "Group the answer by university, then by restaurant. "
    "Include meal type, dish names, serving time, and price when available."
)

# Build the agent with the get_menu tool attached
agent = create_agent(
    model=llm,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_menu]
)

logger.info("Agent ready.")

2026-06-04 04:25:06.205 | INFO     | __main__:<module>:19 - Agent ready.


## Ask the Agent a Question

Change the `question` variable below and run the cell to get an answer.

In [25]:
# Change this question to ask anything about the menu
question = "What is on the menu today? Format the answer in HTML so that I can send it via email."

In [26]:
logger.info("Question: {!r}", question)

# Run the agent with our question
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

# Extract the final text reply from the agent's response messages
answer = ""
for message in reversed(result.get("messages", [])):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        answer = content
        break

logger.info("Answer: {} chars", len(answer))

# Display the answer nicely formatted in the notebook
display(Markdown(answer))

2026-06-04 04:25:06.220 | INFO     | __main__:<module>:1 - Question: 'What is on the menu today? Format the answer in HTML so that I can send it via email.'
2026-06-04 04:25:09.205 | INFO     | getmenus:get_menu:104 - get_menu returned 107 row(s) for 2026-06-04
2026-06-04 04:26:07.778 | INFO     | __main__:<module>:14 - Answer: 17574 chars


<html>
  <body>
    <p>Date: 2026-06-04</p>

    <!-- EWHA Womans University -->
    <h2>Ewha Womans University</h2>
    <h3>Jin·Seon·Mi Building Cafeteria (Jin·Seon·Mi Hall Cafeteria) — campus: Jin·Seon·Mi Hall</h3>
    <ul>
      <li><strong>Lunch</strong>: Mixed Grain Rice — Serving time: 11:00–14:00 — Price: (not listed)</li>
      <li><strong>Dinner</strong>: Mixed Grain Rice — Serving time: 17:00–19:00 — Price: (not listed)</li>
    </ul>

    <!-- Hankuk University of Foreign Studies -->
    <h2>Hankuk University of Foreign Studies</h2>
    <h3>Hankuk University of Foreign Studies Cafeteria</h3>
    <ul>
      <li><strong>Breakfast</strong>: Tuna Kimchi Jjigae (Tuna & Kimchi Stew) — Serving time: 08:00–10:00 — Price: ₩4,000</li>
      <li><strong>Lunch</strong>: Pork Loin Cutlet — Serving time: 11:00–14:30 — Price: ₩4,000</li>
      <li><strong>Lunch</strong>: Vegetable Bibimbap — Serving time: 11:00–14:30 — Price: ₩3,500</li>
      <li><strong>Lunch</strong>: Beef Bulgogi & Mushroom Rice Bowl — Serving time: 11:00–13:30 — Price: ₩6,500</li>
      <li><strong>Dinner</strong>: Spicy Grilled Chicken with Rice — Serving time: 16:40–18:40 — Price: ₩4,000</li>
    </ul>

    <h3>Lunch Noodles (Noodle Corner)</h3>
    <ul>
      <li><strong>Lunch</strong>: Janchi Guksu (Anchovy-based Noodle Soup) — Serving time: 11:00–14:00 — Price: ₩3,000</li>
    </ul>

    <h3>Snack Corner</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: Bowl of Rice — Serving time: 09:00–18:40 — Price: ₩700</li>
      <li><strong>All-day / Unknown meal type</strong>: Cheese (slice) — Serving time: 09:00–18:40 — Price: ₩400</li>
      <li><strong>All-day / Unknown meal type</strong>: Cheese Ramen — Serving time: 09:00–18:40 — Price: ₩2,400</li>
      <li><strong>All-day / Unknown meal type</strong>: Ramen — Serving time: 09:00–18:40 — Price: ₩2,100</li>
      <li><strong>All-day / Unknown meal type</strong>: Ramen with Rice Cake Balls — Serving time: 09:00–18:40 — Price: ₩2,400</li>
      <li><strong>All-day / Unknown meal type</strong>: Toast — Serving time: 09:00–18:40 — Price: ₩1,800</li>
      <li><strong>All-day / Unknown meal type</strong>: Cheese Toast — Serving time: 09:00–18:40 — Price: ₩2,100</li>
    </ul>

    <!-- Hansung University -->
    <h2>Hansung University</h2>
    <h3>K Kitchen Hansang (K Kitchen — Hansang) — campus: K Kitchen Hansang</h3>
    <ul>
      <li><strong>Lunch</strong>: Dakgalbi Rice Bowl (Spicy Stir-fried Chicken Rice Bowl) — Serving time: (not listed) — Price: (not listed)</li>
    </ul>

    <!-- Hanyang University -->
    <h2>Hanyang University</h2>
    <h3>International Dining</h3>
    <ul>
      <li><strong>Lunch</strong>: Giant Pork Cutlet, Cold Buckwheat Noodles (Cold Soba), French Fries, Corn Salad, Furikake Rice, Breakfast Breads (rolls, croissant & butter/jam), Pickles (cucumber & yellow radish), Napa Kimchi — Serving time: (not listed) — Price: (not listed)</li>
    </ul>

    <h3>Korean Dining</h3>
    <ul>
      <li><strong>Dinner</strong>: Smoked Duck & Steamed Napa Cabbage, Soybean Sprout Soup, Sweet Potato Croquette, Imitation Crab Vegetable Stir-fry, Braised Black Beans, Dried Radish Salad, 3 types of Kimchi, Mixed Grain Rice — Serving time: (not listed) — Price: (not listed)</li>
    </ul>

    <h3>Snack / Ramen Stall</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: Sesame Ramen (add rice +₩500) — Serving time: 11:00–19:00 — Price: ₩3,500</li>
    </ul>

    <h3>Life Science Building Cafeteria</h3>
    <ul>
      <li><strong>Lunch</strong>: Daehan Gopchang (Beef Small Intestine Hotpot), Seafood Pancake — Serving time: 11:30–14:00 — Price: ₩12,000</li>
      <li><strong>Lunch</strong>: Giant Pork Cutlet & Cold Soba — Serving time: 11:30–14:00 — Price: ₩12,000</li>
      <li><strong>Lunch</strong>: Smoked Duck Salad (with sauce & drink) — Serving time: 11:30–14:00 — Price: ₩9,000</li>
      <li><strong>Dinner</strong>: Smoked Duck & Steamed Napa Cabbage — Serving time: 16:50–18:20 — Price: ₩12,000</li>
    </ul>

    <h3>New Materials Engineering Building Cafeteria</h3>
    <ul>
      <li><strong>Lunch</strong>: Pork Soft Tofu Stew, Black Rice, Seafood Cutlet — Serving time: 11:30–13:30 — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Shrimp Tempura Rice Bowl & Banquet Noodles — Serving time: 11:30–13:30 — Price: ₩6,500</li>
      <li><strong>Lunch</strong>: Chicken Mayo Rice Bowl — Serving time: 11:30–13:30 — Price: ₩4,700</li>
      <li><strong>Dinner</strong>: Gyeongsang-style Beef Rice Soup (Beef Gukbap) — Serving time: 17:00–18:30 — Price: ₩6,000</li>
    </ul>

    <h3>Today's Cup Rice</h3>
    <ul>
      <li><strong>Unknown meal time</strong>: Stir-fried Brisket & Garlic Scape Bibimbap — Serving time: 16:30–19:00 — Price: ₩4,800</li>
    </ul>

    <h3>Dormitory 2 Cafeteria</h3>
    <ul>
      <li><strong>Breakfast</strong>: Garlic Bagel with Cream Cheese, Carrot Soup, Grilled Sausage with Ketchup, Yogurt/Jelly, Apple Juice — Serving time: 07:30–09:00 — Price: ₩4,800</li>
      <li><strong>Lunch</strong>: Beef Tteokguk-style Noodle Soup, Steamed Meat Dumpling, Fried Fish Cake Skewers, Chive Seasoned Salad, Napa Kimchi — Serving time: 12:00–13:30 — Price: ₩4,800</li>
      <li><strong>Dinner</strong>: Rice, Dried Radish & Soybean Paste Soup, Stir-fried Spicy Pork, Potato & Ham Stir-fry, Dried Radish Salad, Napa Kimchi — Serving time: 17:30–18:50 — Price: ₩4,800</li>
    </ul>

    <h3>Gymnasium Cafeteria</h3>
    <ul>
      <li><strong>Breakfast</strong>: Mini Croissant with Strawberry Jam, Cereal, Milk, Bacon Scramble with Ketchup, Banana, Yogurt, Orange Juice — Serving time: 07:30–08:20 — Price: ₩6,500</li>
      <li><strong>Lunch</strong>: Tteokgalbi Mayo Rice Bowl, Spicy Fish Cake Soup, Chicken Nugget & Rice Cake Sweet & Spicy, Veggie Jjolmyeon, Small Potatoes Braised, Dried Radish Salad, Napa Kimchi — Serving time: 12:00–12:50 — Price: ₩6,500</li>
      <li><strong>Dinner</strong>: Rice, Tuna Kimchi Jjigae, Stir-fried Pork Belly, Pickled Radish Slices, Korean Japchae, Braised Lotus Root, Napa Kimchi, Fruit Pudding — Serving time: 17:30–18:20 — Price: ₩6,500</li>
    </ul>

    <h3>Student Cafeteria (General)</h3>
    <ul>
      <li><strong>Breakfast</strong>: Fish Cake Skewer Soup, Rice, Grilled Thin Beef in Soy Sauce, Seaweed for Lunchbox, Seasoned Dried Anchovy — Serving time: 08:30–09:40 — Price: (not listed)</li>
      <li><strong>Breakfast</strong>: Stir-fried Chicken & Kimchi with Rice, Corn Pancake, Seaweed for Lunchbox, Pickled Soy Sauce Peppers, Cubed Radish Kimchi — Serving time: 08:30–09:40 — Price: ₩1,000</li>
      <li><strong>Breakfast</strong>: Beef & Green Onion Hangover Soup with Rice, Braised Eggs, Seaweed for Lunchbox, Chicory Salad, Cubed Radish Kimchi — Serving time: 08:30–09:40 — Price: (not listed)</li>
      <li><strong>Breakfast (Brunch Set)</strong>: Toast / Mocha Bun, Cabbage Salad & Dressing, Cream Soup / Nuts Mix, Frankfurter & Broccoli-Cauliflower Stir-fry, Banana & Tomato, Seasoned Rice — Serving time: 08:30–09:40 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: [Snack SET] Soup Tteokbokki, Seasoned Rice with Seaweed Flakes, Assorted Fried Items, Steamed Sundae (Korean Sausage), Pickled Radish, Juice — Serving time: 11:30–13:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Gangneung Jang Kalguksu (Spicy Noodle Soup), Rice, Sweet Rice Pancake, Acorn Jelly Salad, Cabbage Salad & Dressing, Cubed Radish Kimchi — Serving time: 11:30–13:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Budae Bokkeum Rice Bowl (Stir-fried Sausage over Rice), Cabbage Salad & Dressing, Cubed Radish Kimchi, Udon Broth — Serving time: 11:30–13:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Spicy Bibim Naengmyeon & Soy Sauce Bulgogi, Seasoned Rice with Seaweed Flakes, Fried Dumplings, Pickled Cucumbers, Yogurt — Serving time: 11:30–13:30 — Price: ₩6,500</li>
      <li><strong>Lunch</strong>: Aqua Donkatsu (Pork Cutlet with Oriental Sauce), Rice, Squid & Octopus Hotbar, Five-Flavor Pickle Salad, Napa Kimchi, Japanese Soup — Serving time: 11:30–13:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Chadol Doenjang Jjigae (Beef Brisket Soybean Paste Stew) — Serving time: 11:00–14:30 — Price: ₩4,800</li>
      <li><strong>Lunch</strong>: Cheese Tonkatsu — Serving time: 11:00–14:30 — Price: ₩4,800</li>
      <li><strong>Lunch</strong>: Chicken Mayo Rice Bowl — Serving time: 11:00–14:30 — Price: ₩4,800</li>
      <li><strong>Lunch</strong>: Hokyon Tonkotsu Ramen (Kihyeon's Tonkotsu Ramen) with Croquette, Rice — Serving time: 11:30–13:30 — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Seafood Yaki Udon — Serving time: 11:30–13:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Smoked Duck Black Bean Fried Rice — Serving time: 11:30–13:30 — Price: ₩4,500</li>
    </ul>

    <!-- Korea University -->
    <h2>Korea University</h2>
    <h3>Science Students' Cafeteria — campus: Natural Sciences Student Cafeteria</h3>
    <ul>
      <li><strong>Dinner</strong>: Chinese-style Fried Rice with Black Bean Sauce, Sweet and Sour Pork — Serving time: 17:30–19:00 — Price: ₩7,000</li>
      <li><strong>Dinner</strong>: Chinese-style Fried Rice & Black Bean Sauce, Sweet & Sour Pork, Egg Pot Soup, Sweet Potato Mash, Juice, Black Rice, Napa Kimchi, Cabbage Salad — Serving time: 17:30–19:00 — Price: ₩7,000</li>
      <li><strong>Breakfast</strong>: Bulgogi Cup Rice / Tuna Mayo Triangle Gimbap — Serving time: 08:00–09:30 — Price: ₩1,000</li>
      <li><strong>Lunch</strong>: Kihyeon's Tonkotsu Ramen & Croquette & Rice, Pickled Radish, Napa Kimchi, Cabbage Salad — Serving time: 11:00–13:30 — Price: ₩6,000</li>
    </ul>

    <h3>Anam Dormitory Cafeteria</h3>
    <ul>
      <li><strong>Breakfast</strong>: Tuna Kimchi Cup Rice or Tuna Triangle Gimbap Set — Serving time: 08:00–09:30 — Price: ₩1,000</li>
      <li><strong>Lunch</strong>: Pumpkin Gochujang Stew Set (with rice and grilled chicken breast & vegetables) — Serving time: 11:30–14:00 — Price: ₩6,000</li>
    </ul>

    <h3>Student Hall Cafeteria (1905 Gohadae Hansang)</h3>
    <ul>
      <li><strong>Breakfast</strong>: Bibim Kalguksu (Mixed Knife-Cut Noodles) — Serving time: 08:00–09:30 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Pork Cutlet Katsu-don — Serving time: 11:00–14:00 — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Rose Braised Chicken & New Jin Noodles — Serving time: 11:00–14:00 — Price: (not listed)</li>
    </ul>

    <!-- Sejong University -->
    <h2>Sejong University</h2>
    <h3>Seasonal Table (Gyejeol Bapsang)</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: 8 side dishes / Soup / Mini Salad Bar — Serving time: 10:30–18:30 — Price: ₩7,000</li>
    </ul>

    <h3>Big Bear Restaurant</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: Steak, Spaghetti, Risotto, Pizza, Salads, etc. — Serving time: 11:30–21:00 — Price: ₩16,000–₩32,000 (source listed price shows 1600032000 — likely a range)</li>
    </ul>

    <h3>Sandeul Food (Sandeul Food Court)</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: Snack items (tteokbokki etc., 15 items) / Western (omelette etc., ~10 items) / Japanese (tonkatsu etc., ~15 items) — Serving time: 10:30–19:00 — Price: ₩6,000</li>
    </ul>

    <h3>Jingwan Kitchen</h3>
    <ul>
      <li><strong>All-day / Unknown meal type</strong>: Cup Rice, Tuna Mayo Bibimbap, Soft Tofu Stew, Raw Fish Bibimbap, Cheese Tonkatsu, ~30 other menu items — Serving time: 09:00–19:00 — Price: ₩2,800–₩5,800</li>
    </ul>

    <!-- Seoul National University -->
    <h2>Seoul National University</h2>
    <h3>Cafeteria Dongwon Bldg I-113</h3>
    <ul>
      <li><strong>Lunch</strong>: Bean Paste Stew with Snails & Shrimp — Price: (not listed)</li>
      <li><strong>Lunch</strong>: Green Onion Bulgogi — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Tteokbokki — Price: ₩6,000</li>
    </ul>

    <h3>Cafeteria No.1 (Student Center) — Bldg. C-63</h3>
    <ul>
      <li><strong>Lunch</strong>: Double Hamburger Steak with Cheese & French Fries — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Green Onion Bulgogi — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Kimchi Stew with Pork Belly — Price: ₩5,500</li>
      <li><strong>Lunch</strong>: Rich Soybean Paste Stew — Price: ₩3,000</li>
      <li><strong>Lunch</strong>: Pork Cutlet Set Menu — Price: ₩3,000</li>
      <li><strong>Lunch</strong>: Spicy Soft Tofu Stew with Clam — Price: ₩3,000</li>
    </ul>

    <h3>Dormitory Cafeteria — Bldg. J-919</h3>
    <ul>
      <li><strong>Lunch</strong>: Rice topped with Spam & Mayonnaise & Tteokbokki-style Fish Cakes and Quail Eggs — Price: ₩6,000</li>
      <li><strong>Lunch</strong>: Vietnamese Rice Noodle with Beef (Pho-style) — Price: ₩6,500</li>
    </ul>

    <h3>Engineering & Research Center 2 Cafeteria — Bldg. F-302</h3>
    <ul>
      <li><strong>Breakfast</strong>: Buffet — Price: ₩6,500</li>
      <li><strong>Lunch</strong>: Pork & Kimchi Stew — Price: (not listed)</li>
      <li><strong>Dinner</strong>: Jalapeño & Sabi Mayo Chicken — Price: (not listed)</li>
    </ul>

    <h3>Gap Jjigae (Gat Jjigae)</h3>
    <ul>
      <li><strong>Unknown meal type</strong>: Soft Tofu Stew (includes rice) — Price: ₩6,000</li>
      <li><strong>Unknown meal type</strong>: Beef Brisket Soft Tofu Stew (includes rice) — Price: ₩6,900</li>
    </ul>

    <h3>Babi DdeunDeun (Campus Quick Service)</h3>
    <ul>
      <li><strong>Unknown meal type</strong>: Fried Egg — Price: ₩800</li>
      <li><strong>Unknown meal type</strong>: Stir-fried Kimchi Rice Bowl — Price: ₩5,800</li>
      <li><strong>Unknown meal type</strong>: Spam Mayo — Price: ₩3,900</li>
      <li><strong>Unknown meal type</strong>: Meat Rice Bowl — Price: ₩3,900</li>
      <li><strong>Unknown meal type</strong>: Tuna Mayo — Price: ₩3,500</li>
      <li><strong>Unknown meal type</strong>: Tuna Mayo (R) + Spicy Pork One-dish Set — Price: ₩6,500</li>
    </ul>

    <h3>Arts Cafeteria</h3>
    <ul>
      <li><strong>Lunch</strong>: Ttukbaegi Tteokbokki with Ramen, Fish Cake Skewer Soup, Fried Squid Legs & Gim-mari (seaweed roll) — Price: ₩6,500 — Serving time: 11:30–14:00</li>
      <li><strong>Lunch</strong>: Self-serve Ramen, Rice, Kimchi — Price: ₩3,000 — Serving time: 11:30–13:00</li>
    </ul>

    <h3>Jahayeon Cafeteria — 2nd Floor</h3>
    <ul>
      <li><strong>Dinner</strong>: Old-fashioned Fried Chicken — Price: ₩6,000 — Serving time: 17:00–18:30</li>
      <li><strong>Lunch</strong>: Dumpling Short Rib Soup (Mandu Galbitang) — Price: ₩6,000 — Serving time: 11:30–14:00</li>
    </ul>

    <h3>Jahayeon Cafeteria — 3rd Floor</h3>
    <ul>
      <li><strong>Lunch</strong>: Healthy Summer Meal Vol.1 — “Wrap Day” (special set) — Price: ₩13,000 — Serving time: 11:30–14:00</li>
    </ul>

    <h3>Kitchen101</h3>
    <ul>
      <li><strong>All-day</strong>: Salmon Bibimbap (Large) — Price: ₩8,900 — Serving time: 09:30–19:00</li>
      <li><strong>All-day</strong>: Kitchen101 Vegetable Bibimbap — Price: ₩4,900 — Serving time: 09:30–19:00</li>
    </ul>

    <h3>PoPo420 (Vietnamese Corner)</h3>
    <ul>
      <li><strong>All-day</strong>: Mixed Rice Noodles — Price: ₩6,500</li>
      <li><strong>All-day</strong>: Popo Rice Noodles — Price: ₩4,500</li>
    </ul>

    <h3>Student Hall Cafeteria</h3>
    <ul>
      <li><strong>Breakfast</strong>: Cheonggukjang Stew — Price: ₩3,000 — Serving time: 08:00–10:00</li>
      <li><strong>Lunch</strong>: Pork Cutlet Set — Price: ₩3,000 — Serving time: 11:00–14:30</li>
      <li><strong>Lunch</strong>: Scallion Bulgogi — Price: ₩6,000</li>
      <li><strong>Dinner</strong>: Double Cheese Hamburg Steak & French Fries — Price: ₩6,000 — Serving time: 17:00–19:00</li>
    </ul>

    <!-- SKKU (Sungkyunkwan University) -->
    <h2>Sungkyunkwan University (SKKU)</h2>
    <h3>Faculty Cafeteria — Faculty Dining</h3>
    <ul>
      <li><strong>Lunch</strong>: Soft Tofu Stew, Soy Sauce Pork Bulgogi, Scrambled Egg, Kimchi Pancake, Square Fish Cake Stir-fry, Lotus Root with Sesame Sauce, Pickled Mustard Greens, Chicory Salad, Black Rice / Brown Rice, 2 kinds of Kimchi, Green Salad, Bread, Seaweed, Cup Noodle — Serving time: 11:20–13:30 — Price: ₩9,000</li>
      <li><strong>Lunch (Corner)</strong>: Same menu as above (Faculty Corner) — Serving time: 11:20–13:30 — Price: ₩9,000</li>
    </ul>

    <!-- Soongsil University -->
    <h2>Soongsil University</h2>
    <h3>Dormitory Cafeteria</h3>
    <ul>
      <li><strong>Lunch</strong>: Fish Cake Soup, Rice & Black Rice, Spicy Braised Chicken with Cheese (Snowflake Cheese Style) — Serving time: 11:00–13:50 — Price: (not listed)</li>
      <li><strong>Dinner</strong>: Pork Mapo Tofu (over rice), Shrimp Tempura, Bean Sprout Soup, Chili Takoyaki, Coleslaw, Napa Kimchi, Yogurt — Serving time: 17:00–18:30 — Price: (not listed)</li>
      <li><strong>Dinner</strong>: Pork Mapo Tofu Rice Bowl (single-item listing) — Serving time: 17:00–18:30 — Price: (not listed)</li>
    </ul>

    <!-- Notes / Sources -->
    <hr/>
    <p>
      Sources: menus scraped from each university's cafeteria sites (source pages available for each item).
      If you want, I can:
    </p>
    <ul>
      <li>Include direct links to each menu item’s source page,</li>
      <li>Filter by a specific university or meal (e.g., only lunch items), or</li>
      <li>Export this as an email-ready HTML file attachment.</li>
    </ul>
  </body>
</html>

In [ ]:
import httpx

def send_html_via_smtp2go(html_body: str, subject: str) -> dict:
    # Log function start so the step appears in logs/
    logger.info("Starting SMTP2GO email send function.")

    # Reload .env values (safe in notebooks) in case variables changed
    load_dotenv()

    # Read SMTP2GO configuration from environment variables
    smtp2go_api_key = os.getenv("SMTP2GO_API_KEY")
    smtp2go_sender_email = os.getenv("SMTP2GO_SENDER_EMAIL")
    smtp2go_recipient_email = os.getenv("SMTP2GO_RECIPIENT_EMAIL")

    # Validate required configuration before sending
    if not smtp2go_api_key:
        logger.error("Missing SMTP2GO_API_KEY in .env")
        raise ValueError("SMTP2GO_API_KEY is missing in .env")
    if not smtp2go_sender_email:
        logger.error("Missing SMTP2GO_SENDER_EMAIL in .env")
        raise ValueError("SMTP2GO_SENDER_EMAIL is missing in .env")
    if not smtp2go_recipient_email:
        logger.error("Missing SMTP2GO_RECIPIENT_EMAIL in .env")
        raise ValueError("SMTP2GO_RECIPIENT_EMAIL is missing in .env")

    logger.info("SMTP2GO configuration loaded successfully.")
    # Masked key for diagnostics (don't log full key)
    try:
        masked = smtp2go_api_key[:4] + '...' + smtp2go_api_key[-4:] if smtp2go_api_key else '****'
    except Exception:
        masked = '****'
    logger.info("SMTP2GO API key (masked): {}", masked)

    # SMTP2GO API endpoint and required headers
    endpoint_url = "https://api.smtp2go.com/v3/email/send"
    headers = {
        "Content-Type": "application/json",
        "accept": "application/json",
        "X-Smtp2go-Api-Key": smtp2go_api_key
    }

    # Build the email payload using html_body (include api_key for compatibility)
    payload = {
        "api_key": smtp2go_api_key,
        "sender": smtp2go_sender_email,
        "to": [smtp2go_recipient_email],
        "subject": subject,
        "html_body": html_body
    }

    logger.info(
        "Sending email via SMTP2GO. sender='{}' recipient='{}' subject='{}'",
        smtp2go_sender_email,
        smtp2go_recipient_email,
        subject
    )

    try:
        # Send the POST request to SMTP2GO
        response = httpx.post(endpoint_url, headers=headers, json=payload, timeout=30.0)
        logger.info("SMTP2GO response status code: {}", response.status_code)

        # Raise an exception for 4xx/5xx responses
        response.raise_for_status()

        # Parse JSON response and log result summary
        response_data = response.json()
        logger.info("SMTP2GO response parsed successfully.")
        logger.info("SMTP2GO request_id: {}", response_data.get("request_id"))

        data_block = response_data.get("data", {})
        logger.info(
            "SMTP2GO send result: succeeded={} failed={}",
            data_block.get("succeeded"),
            data_block.get("failed")
        )

        return response_data

    except httpx.HTTPStatusError as error:
        # Log detailed API error body if present
        logger.exception("SMTP2GO returned an HTTP error: {}", str(error))
        try:
            logger.error("SMTP2GO error response body: {}", error.response.text)
        except Exception:
            logger.error("SMTP2GO error response body: <unavailable>")
        raise
    except Exception as error:
        # Log unexpected errors
        logger.exception("Unexpected error while sending SMTP2GO email: {}", str(error))
        raise

In [28]:
send_html_via_smtp2go(answer, subject="Today's Campus Menu")

2026-06-04 04:32:09.825 | INFO     | __main__:send_html_via_smtp2go:5 - Starting SMTP2GO email send function.
2026-06-04 04:32:09.829 | INFO     | __main__:send_html_via_smtp2go:26 - SMTP2GO configuration loaded successfully.
2026-06-04 04:32:09.830 | INFO     | __main__:send_html_via_smtp2go:44 - Sending email via SMTP2GO. sender='p.i.m@skku.edu' recipient='p.i.m@g.skku.edu' subject='Today's Campus Menu'
2026-06-04 04:32:11.191 | INFO     | __main__:send_html_via_smtp2go:54 - SMTP2GO response status code: 400
2026-06-04 04:32:11.192 | ERROR    | __main__:send_html_via_smtp2go:75 - SMTP2GO returned an HTTP error: Client error '400 Bad Request' for url 'https://api.smtp2go.com/v3/email/send'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400
Traceback (most recent call last):

  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/workspaces/hai5016-project/.venv/lib/python3.11/site-packages

HTTPStatusError: Client error '400 Bad Request' for url 'https://api.smtp2go.com/v3/email/send'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400